# Load Data

In [ ]:
import pandas as pd
import os

data_folder = 'data/'

for file in sorted(os.listdir(data_folder)):
    if file.endswith('.csv'):
        df = pd.read_csv(os.path.join(data_folder, file))
        print(f"\n{'='*60}")
        print(f"📄 {file} — {df.shape[0]} rows, {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
        print(df.head(2))
    

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
print("✅ Libraries loaded")

In [ ]:
# Cell 2: Load all datasets
data_folder = 'data/'

# Transaction data
abm = pd.read_csv(f'{data_folder}abm.csv')
card = pd.read_csv(f'{data_folder}card.csv')
cheque = pd.read_csv(f'{data_folder}cheque.csv')
eft = pd.read_csv(f'{data_folder}eft.csv')
emt = pd.read_csv(f'{data_folder}emt.csv')
westernunion = pd.read_csv(f'{data_folder}westernunion.csv')
wire = pd.read_csv(f'{data_folder}wire.csv')

# Customer profiles
kyc_individual = pd.read_csv(f'{data_folder}kyc_individual.csv')
kyc_smallbusiness = pd.read_csv(f'{data_folder}kyc_smallbusiness.csv')

# Reference tables
industry_codes = pd.read_csv(f'{data_folder}kyc_industry_codes.csv')
occupation_codes = pd.read_csv(f'{data_folder}kyc_occupation_codes.csv')

# Labels (target variable)
labels = pd.read_csv(f'{data_folder}labels.csv')

print(f"✅ All data loaded")
print(f"\nLabels distribution:\n{labels['label'].value_counts()}")
print(f"\nFraud rate: {labels['label'].mean()*100:.1f}%")

In [ ]:
# Cell 3: Who are the labeled customers?
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
labels['label'].value_counts().plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_title('Label Distribution (0 = Legit, 1 = Fraud)', fontsize=14)
ax.set_xlabel('Label')
ax.set_ylabel('Count')
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
for i, v in enumerate(labels['label'].value_counts().values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Check if labeled customers are individuals, businesses, or both
labeled_ids = set(labels['customer_id'])
in_individual = labeled_ids & set(kyc_individual['customer_id'])
in_business = labeled_ids & set(kyc_smallbusiness['customer_id'])
print(f"Labeled customers found in kyc_individual: {len(in_individual)}")
print(f"Labeled customers found in kyc_smallbusiness: {len(in_business)}")
print(f"Total labeled: {len(labels)}")

In [ ]:
# Cell 4: Build transaction features per customer

transaction_tables = {
    'abm': abm,
    'card': card,
    'cheque': cheque,
    'eft': eft,
    'emt': emt,
    'westernunion': westernunion,
    'wire': wire
}

def build_txn_features(df, prefix):
    """Aggregate transaction-level data into customer-level features."""
    agg = df.groupby('customer_id').agg(
        txn_count=('transaction_id', 'count'),
        total_amount=('amount_cad', 'sum'),
        avg_amount=('amount_cad', 'mean'),
        max_amount=('amount_cad', 'max'),
        min_amount=('amount_cad', 'min'),
        std_amount=('amount_cad', 'std'),
    ).reset_index()
    
    # Debit vs Credit counts
    if 'debit_credit' in df.columns:
        dc = df.groupby('customer_id')['debit_credit'].value_counts().unstack(fill_value=0)
        dc.columns = [f'{prefix}_debit_count' if c == 'D' else f'{prefix}_credit_count' for c in dc.columns]
        dc = dc.reset_index()
        agg = agg.merge(dc, on='customer_id', how='left')
    
    # Rename columns with prefix
    agg.columns = ['customer_id'] + [f'{prefix}_{c}' for c in agg.columns[1:]]
    
    return agg

# Build features for each transaction type
all_features = []
for name, df in transaction_tables.items():
    feats = build_txn_features(df, name)
    all_features.append(feats)
    print(f"✅ {name}: {feats.shape[1]-1} features for {feats.shape[0]} customers")

# Merge all transaction features together
txn_features = all_features[0]
for feat_df in all_features[1:]:
    txn_features = txn_features.merge(feat_df, on='customer_id', how='outer')

print(f"\n✅ Combined transaction features: {txn_features.shape}")

In [ ]:
# Cell 5: Card-specific features (these are unique and valuable)

# Ecommerce ratio per customer
card_extra = card.groupby('customer_id').agg(
    card_ecommerce_ratio=('ecommerce_ind', 'mean'),
    card_unique_merchants=('merchant_category', 'nunique'),
    card_unique_countries=('country', 'nunique'),
    card_unique_cities=('city', 'nunique'),
).reset_index()

# International transaction ratio for card
card['is_international'] = (card['country'] != 'CA').astype(int)
intl = card.groupby('customer_id')['is_international'].mean().reset_index()
intl.columns = ['customer_id', 'card_intl_ratio']

card_extra = card_extra.merge(intl, on='customer_id', how='left')

# Merge into main features
txn_features = txn_features.merge(card_extra, on='customer_id', how='left')

print(f"✅ Card-specific features added. Shape: {txn_features.shape}")

In [ ]:
# Cell 6: KYC profile features

# Individual customers
kyc_ind = kyc_individual.copy()
kyc_ind['birth_date'] = pd.to_datetime(kyc_ind['birth_date'], errors='coerce')
kyc_ind['onboard_date'] = pd.to_datetime(kyc_ind['onboard_date'], errors='coerce')
kyc_ind['age'] = (pd.Timestamp('2024-11-01') - kyc_ind['birth_date']).dt.days / 365.25
kyc_ind['account_age_years'] = (pd.Timestamp('2024-11-01') - kyc_ind['onboard_date']).dt.days / 365.25

# Encode gender
kyc_ind['gender_encoded'] = kyc_ind['gender'].map({'MALE': 0, 'FEMALE': 1}).fillna(-1)
kyc_ind['marital_encoded'] = kyc_ind['marital_status'].map({
    'Single': 0, 'Married': 1, 'Common-Law': 2, 'Divorced': 3, 'Widowed': 4, 'Separated': 5
}).fillna(-1)

kyc_ind_features = kyc_ind[['customer_id', 'age', 'account_age_years', 'income', 
                             'gender_encoded', 'marital_encoded', 'occupation_code']].copy()

# Small business customers
kyc_biz = kyc_smallbusiness.copy()
kyc_biz['established_date'] = pd.to_datetime(kyc_biz['established_date'], errors='coerce')
kyc_biz['onboard_date'] = pd.to_datetime(kyc_biz['onboard_date'], errors='coerce')
kyc_biz['business_age_years'] = (pd.Timestamp('2024-11-01') - kyc_biz['established_date']).dt.days / 365.25
kyc_biz['biz_account_age_years'] = (pd.Timestamp('2024-11-01') - kyc_biz['onboard_date']).dt.days / 365.25

kyc_biz_features = kyc_biz[['customer_id', 'business_age_years', 'biz_account_age_years',
                              'employee_count', 'sales', 'industry_code']].copy()

# Combine KYC (individuals + businesses)
# Add a flag for customer type
kyc_ind_features['is_business'] = 0
kyc_biz_features['is_business'] = 1

# Rename overlapping columns for merge
kyc_biz_features = kyc_biz_features.rename(columns={
    'business_age_years': 'age', 
    'biz_account_age_years': 'account_age_years',
    'sales': 'income',
    'employee_count': 'employee_count'
})

kyc_all = pd.concat([kyc_ind_features, kyc_biz_features], ignore_index=True)

# Merge KYC with transaction features
full_features = txn_features.merge(kyc_all, on='customer_id', how='left')

print(f"✅ KYC features merged. Full feature set: {full_features.shape}")

In [ ]:
# Cell 7: Create the modeling dataset

# Only keep customers that have labels
df_model = full_features.merge(labels, on='customer_id', how='inner')

print(f"Modeling dataset: {df_model.shape[0]} customers, {df_model.shape[1]} columns")
print(f"Fraud distribution:\n{df_model['label'].value_counts()}")
print(f"\nMissing values per column:")
print(df_model.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
# Cell 8: Clean up for modeling

# Drop customer_id (not a feature) and separate target
X = df_model.drop(columns=['customer_id', 'label'])
y = df_model['label']

# Fill NaN with 0 for transaction features (no transactions = 0 activity)
# This makes sense: if a customer has no wire transfers, their wire features should be 0
X = X.fillna(0)

# Check final shape
print(f"✅ Features: {X.shape}")
print(f"✅ Target: {y.shape}")
print(f"✅ Feature names ({len(X.columns)}):")
for i, col in enumerate(X.columns):
    print(f"  {i+1}. {col}")

In [ ]:
# Cell 8b: Fix non-numeric columns

# Check for any non-numeric columns still in X
non_numeric = X.select_dtypes(include=['object']).columns.tolist()
print(f"Non-numeric columns found: {non_numeric}")

# Drop them (occupation_code has strings like 'RETIRED' — not worth encoding for now)
X = X.drop(columns=non_numeric)

print(f"✅ Cleaned features: {X.shape}")

# Re-do the train/test split with cleaned X
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
# Cell 9: Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train fraud rate: {y_train.mean()*100:.1f}%")
print(f"Test fraud rate: {y_test.mean()*100:.1f}%")

In [ ]:
# Cell 10: Train and compare models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

# Scale features for logistic regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42)
}

results = {}
for name, model in models.items():
    # Use scaled data for LR, raw for tree models
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model': model, 'y_pred': y_pred, 'y_prob': y_prob, 'auc': auc}
    
    print(f"\n{'='*50}")
    print(f"📊 {name} (AUC: {auc:.4f})")
    print(classification_report(y_test, y_pred))

In [ ]:
# Cell 11: ROC Curve Comparison
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

colors = ['#3498db', '#2ecc71', '#e74c3c']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Model Comparison', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Confusion Matrix for Best Model
best_name = max(results, key=lambda k: results[k]['auc'])
best_result = results[best_name]
print(f"🏆 Best model: {best_name} (AUC: {best_result['auc']:.4f})")

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
cm = confusion_matrix(y_test, best_result['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Legitimate', 'Fraud'], yticklabels=['Legitimate', 'Fraud'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Feature Importance
best_model = results[best_name]['model']

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
else:
    importances = np.abs(best_model.coef_[0])

feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': importances
}).sort_values('importance', ascending=True).tail(15)

fig, ax = plt.subplots(1, 1, figsize=(10, 7))
ax.barh(feat_imp['feature'], feat_imp['importance'], color='#3498db')
ax.set_title(f'Top 15 Features — {best_name}', fontsize=14)
ax.set_xlabel('Importance', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Automated Fraud Explanation Generator

def explain_flag(row, feature_importances, feature_names, top_n=5):
    """Generate a human-readable explanation for why a customer was flagged."""
    
    # Get the top contributing features for this customer
    contributions = []
    for feat, imp in zip(feature_names, feature_importances):
        val = row[feat]
        if val != 0:
            contributions.append((feat, imp, val))
    
    # Sort by feature importance
    contributions.sort(key=lambda x: x[1], reverse=True)
    top_contribs = contributions[:top_n]
    
    # Generate explanation
    explanation_parts = []
    for feat, imp, val in top_contribs:
        # Make feature names human-readable
        readable = feat.replace('_', ' ').replace('txn count', 'transaction count')
        explanation_parts.append(f"• {readable}: {val:,.2f}")
    
    explanation = "⚠️ FLAGGED — Key risk indicators:\n"
    explanation += "\n".join(explanation_parts)
    
    return explanation

# Demo: explain a flagged customer
flagged_indices = y_test[y_test == 1].index.tolist()

if flagged_indices:
    # Get feature importances from best model
    if hasattr(best_model, 'feature_importances_'):
        feat_imp_values = best_model.feature_importances_
    else:
        feat_imp_values = np.abs(best_model.coef_[0])
    
    print("=" * 60)
    print("AUTOMATED FRAUD EXPLANATION REPORT")
    print("=" * 60)
    
    for idx in flagged_indices[:3]:  # Show 3 examples
        customer_row = X.loc[idx]
        customer_id = df_model.loc[idx, 'customer_id']
        explanation = explain_flag(customer_row, feat_imp_values, X.columns)
        
        print(f"\n🔍 Customer: {customer_id}")
        print(explanation)
        print("-" * 40)

In [ ]:
# Cell 15: Project Summary Stats (for your pitch deck)

total_transactions = sum([len(df) for df in transaction_tables.values()])
total_customers = len(full_features)
labeled_customers = len(labels)
num_features = X.shape[1]
best_auc = results[best_name]['auc']
fraud_count = labels['label'].sum()
legit_count = len(labels) - fraud_count

print("=" * 60)
print("📊 PROJECT SUMMARY — IMI BIG DATA & AI COMPETITION")
print("=" * 60)
print(f"Total transactions analyzed: {total_transactions:,}")
print(f"Total unique customers: {total_customers:,}")
print(f"Labeled customers: {labeled_customers}")
print(f"  - Legitimate: {legit_count}")
print(f"  - Fraudulent: {fraud_count}")
print(f"Fraud rate: {labels['label'].mean()*100:.1f}%")
print(f"Features engineered: {num_features}")
print(f"Transaction types: {len(transaction_tables)}")
print(f"Best model: {best_name}")
print(f"Best AUC: {best_auc:.4f}")
print(f"Data sources: 12 CSV files from partner bank")

# Baseline

In [ ]:
# Baseline comparison
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_prob = dummy.predict_proba(X_test)[:, 1] if hasattr(dummy, 'predict_proba') else np.zeros(len(y_test))

# Also compare against a simple logistic regression with NO feature engineering
# Your best model vs logistic regression = improvement from better modeling
lr_auc = results['Logistic Regression']['auc']
best_auc = results[best_name]['auc']
improvement = ((best_auc - lr_auc) / lr_auc) * 100

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Best Model ({best_name}) AUC: {best_auc:.4f}")
print(f"Improvement: {improvement:.1f}%")